# 7.3 Transformer Encoder

這份 Notebook 建立小型 Transformer encoder，並用在三類數值序列分類任務。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)


## 2. 建立序列分類資料


In [ ]:
CLASS_NAMES = np.array(['trend', 'periodic', 'spike'])
SEQ_LEN = 32
FEATURES = 2

def generate_sequence(label, rng):
    t = np.linspace(0, 1, SEQ_LEN)
    noise = rng.normal(0, 0.08, SEQ_LEN)
    if label == 0:
        value = 1.5 * t + 0.2 * np.sin(2 * np.pi * t) + noise
    elif label == 1:
        value = np.sin(2 * np.pi * 3 * t + rng.uniform(-0.3, 0.3)) + noise
    else:
        center = rng.integers(SEQ_LEN // 4, SEQ_LEN * 3 // 4)
        value = noise + 2.0 * np.exp(-0.5 * ((np.arange(SEQ_LEN) - center) / 2.2) ** 2)
    delta = np.diff(value, prepend=value[0])
    return np.stack([value, delta], axis=-1).astype('float32')

def make_sequence_dataset(samples_per_class=220, seed=42):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for label in range(len(CLASS_NAMES)):
        for _ in range(samples_per_class):
            X.append(generate_sequence(label, rng))
            y.append(label)
    X = np.stack(X).astype('float32')
    y = np.array(y, dtype='int64')
    idx = rng.permutation(len(y))
    return X[idx], y[idx]

def split_and_standardize(X, y):
    x_train, x_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    x_valid, x_test, y_valid, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
    mean = x_train.mean(axis=(0, 1), keepdims=True)
    std = x_train.std(axis=(0, 1), keepdims=True) + 1e-8
    return (x_train - mean) / std, y_train, (x_valid - mean) / std, y_valid, (x_test - mean) / std, y_test

X, y = make_sequence_dataset(samples_per_class=220, seed=8)
x_train, y_train, x_valid, y_valid, x_test, y_test = split_and_standardize(X, y)
print(x_train.shape, x_valid.shape, x_test.shape)


## 3. 定義 position embedding 與 encoder block


In [ ]:
class PositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, embed_dim):
        super().__init__()
        self.position_embedding = tf.keras.layers.Embedding(input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length = sequence_length

    def call(self, inputs):
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        return inputs + self.position_embedding(positions)

class TransformerEncoderBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation='relu'),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(dropout)
        self.dropout2 = tf.keras.layers.Dropout(dropout)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


## 4. 建立 Transformer encoder 分類模型


In [ ]:
EMBED_DIM = 32
NUM_HEADS = 2
FF_DIM = 64

inputs = tf.keras.Input(shape=(SEQ_LEN, FEATURES))
x = tf.keras.layers.Dense(EMBED_DIM)(inputs)
x = PositionEmbedding(SEQ_LEN, EMBED_DIM)(x)
x = TransformerEncoderBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


## 5. 訓練與評估


In [ ]:
history = model.fit(x_train, y_train, validation_data=(x_valid, y_valid), epochs=16, batch_size=32, verbose=0)
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='Transformer encoder accuracy')
plt.ylim(0, 1.05)
plt.show()
y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
print(model.evaluate(x_test, y_test, verbose=0, return_dict=True))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


## 6. 如何套用自己的資料

把資料整理成 `(samples, sequence_length, features)`。數值序列可先用 Dense 投影到 embedding 維度；文字序列則通常先使用 token embedding。


## 7. 小結

Transformer encoder block 將 self-attention、feed-forward、residual connection 與 layer normalization 組合成可訓練模型。
